# Prédiction du taux de réussite au Baccalauréat

1. **Modèle de référence (baseline)** — le plus simple possible pour avoir un point de comparaison  
2. **Prétraitement progressif** — on enrichit pas à pas et on mesure le gain réel  
3. **Comparaison équitable via Pipeline + cross-validation**  
4. **Sélection et optimisation** du meilleur modèle

##### Récapitulatif du process dans la partie **12.Bilan**.

### Problème: *Ce groupe de candidats est-il à risque?*
--> Un proviseur ou référent académique prépare la session du bac.
Il sélectionne les caractéristiques (académie, série, sexe, statut du candidat) et le modèle réponds avec le taux de réussite attendu pour ce groupe.

#### Permet de:
- identifier avant les épreuves les groupes structurellement défavorisés.
- orienter les dispositifs de soutien (heures supp, etc...) là où le besoin est prévisible.
- comparer un groupe à d'autres groupes pour contextualiser

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.inspection import permutation_importance

import joblib

import warnings
warnings.filterwarnings('ignore')

print("Librairies importées")

Librairies importées


## 2. Chargement et Analyses

In [2]:
df_raw = pd.read_csv('data/fr-en-baccalaureat-par-academie.csv', sep=';', encoding='utf-8-sig')
print(f"Dimensions : {df_raw.shape[0]:,} lignes x {df_raw.shape[1]} colonnes")
df_raw.head()

Dimensions : 43,217 lignes x 22 colonnes


,Session,Code académie,Libellé académie,Sexe,Statut du candidat,Voie,Série,Diplôme spécialité,Nombre d'inscrits,Nombre de présents,...,"Nombre d'ajournés, passant les épreuves du 2nd groupe",Nombre d'admis à l'issue du 2nd groupe,Nombre de refusés à l'issue du 2nd groupe,Nombre d'admis totaux,Nombre d'admis avec mention TB avec les félicitations du jury,Nombre d'admis avec mention TB sans les félicitations du jury,Nombre d'admis avec mention B,Nombre d'admis avec mention AB,Nombre d'admis sans mention,Nombre de refusés totaux
0,2021,4,BORDEAUX,FEMININ,SCOLAIRE,BAC PROFESSIONNEL,BAC PRO SERV,BAC PRO 33004 ACC.SOINS-S.PERS. OPT.EN STRUCTUR,471,470,...,8,4,4,453,0,29,164,170,90,17
1,2021,4,BORDEAUX,FEMININ,SCOLAIRE,BAC PROFESSIONNEL,BAC PRO SERV,BAC PRO 34304 HYGIENE PROPRETE STERILISATION,10,10,...,0,0,0,7,0,0,0,5,2,3
2,2021,4,BORDEAUX,FEMININ,SCOLAIRE,BAC PROFESSIONNEL,BAC PRO SERV,BAC PRO AG 33002 SERV PERS TERRIT,419,409,...,0,0,0,394,0,22,76,142,154,15
3,2021,4,BORDEAUX,FEMININ,SCOLAIRE,BAC TECHNOLOGIQUE,BAC STD2A,BAC STD2A SCIENCES & TECHNO. DESIGN-ARTS APPLI...,178,178,...,4,2,2,176,2,10,70,69,25,2
4,2021,4,BORDEAUX,FEMININ,SCOLAIRE,BAC TECHNOLOGIQUE,BAC STL,BAC STL BIOCHIM.-BIOLOGIE-BIOTECHNOL.,143,143,...,5,4,1,142,0,9,27,49,57,1


In [3]:
print("Types de données :")
print(df_raw.dtypes)
print()
print("Valeurs manquantes :")
print(df_raw.isnull().sum())

Types de données :
Session                                                           int64
Code académie                                                     int64
Libellé académie                                                 object
Sexe                                                             object
Statut du candidat                                               object
Voie                                                             object
Série                                                            object
Diplôme spécialité                                               object
Nombre d'inscrits                                                 int64
Nombre de présents                                                int64
Nombre d'admis au 1er groupe                                      int64
Nombre de refusés au 1er groupe                                   int64
Nombre d'ajournés, passant les épreuves du 2nd groupe             int64
Nombre d'admis à l'issue du 2nd groupe       

In [4]:
df_raw.describe(include='all')

,Session,Code académie,Libellé académie,Sexe,Statut du candidat,Voie,Série,Diplôme spécialité,Nombre d'inscrits,Nombre de présents,...,"Nombre d'ajournés, passant les épreuves du 2nd groupe",Nombre d'admis à l'issue du 2nd groupe,Nombre de refusés à l'issue du 2nd groupe,Nombre d'admis totaux,Nombre d'admis avec mention TB avec les félicitations du jury,Nombre d'admis avec mention TB sans les félicitations du jury,Nombre d'admis avec mention B,Nombre d'admis avec mention AB,Nombre d'admis sans mention,Nombre de refusés totaux
count,43217.000000,43217.000000,43217,43217,43217,43217,43217,43217,43217.000000,43217.000000,...,43217.000000,43217.000000,43217.000000,43217.000000,43217.000000,43217.000000,43217.000000,43217.000000,43217.000000,43217.000000
unique,NaN,NaN,30,4,5,3,23,310,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NANTES,MASCULIN,SCOLAIRE,BAC PROFESSIONNEL,BAC PRO PROD,BAC GENE 10019 BAC GENERAL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,1972,19307,23133,34830,18359,715,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2023.026494,17.566027,NaN,NaN,NaN,NaN,NaN,NaN,86.353079,85.840803,...,6.269061,4.376981,1.892080,78.667075,0.831478,6.765324,16.891964,26.374343,27.803966,7.173728
std,1.413073,14.360495,NaN,NaN,NaN,NaN,NaN,NaN,647.273492,647.234306,...,44.633740,31.967000,12.986692,624.645612,11.837864,80.729542,161.849247,202.235609,179.021704,28.379103
min,2021.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2022.000000,9.000000,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,2.000000,...,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2023.000000,15.000000,NaN,NaN,NaN,NaN,NaN,NaN,8.000000,7.000000,...,0.000000,0.000000,0.000000,6.000000,0.000000,0.000000,1.000000,2.000000,2.000000,1.000000
75%,2024.000000,23.000000,NaN,NaN,NaN,NaN,NaN,NaN,29.000000,28.000000,...,1.000000,1.000000,0.000000,24.000000,0.000000,1.000000,5.000000,8.000000,10.000000,4.000000


**Pas de valeurs manquantes.**

In [5]:
# Heatmap Voie × Série — montrer que chaque série appartient à une seule voie
cross = pd.crosstab(df_raw['Série'], df_raw['Voie'])
cross_bin = (cross > 0).astype(int)

fig = px.imshow(
    cross_bin,
    color_continuous_scale=[[0, '#F0F0F0'], [1, '#2E6DA4']],
    title='Appartenance Série → Voie (1 = combinaison existante)',
    labels={'x': 'Voie', 'y': 'Série', 'color': ''},
    aspect='auto'
)
fig.update_layout(height=550, coloraxis_showscale=False)
fig.show()

## 3. Construction de la variable cible

Le **taux de réussite** est défini comme :

`taux_reussite = (Nombre d'admis totaux / Nombre de présents) * 100`

On élimine les lignes où `Nombre de présents == 0` (groupes vides, division par zéro).

### Agrégation par groupe structurel

Sans `Diplôme spécialité`, plusieurs lignes représentent le **même groupe structurel** (même Série × Académie × Sexe × Statut) avec des spécialités différentes. On les regroupe en sommant présents et admis avant de recalculer le taux — ce qui est plus correct que de faire une moyenne de taux.

In [6]:
df = df_raw[df_raw['Nombre de présents'] > 0].copy()
print(f"Lignes brutes (présents > 0) : {len(df):,}")

# Harmonisation avant agrégation
df['Sexe'] = df['Sexe'].replace({'FILLES': 'FEMININ', 'GARCONS': 'MASCULIN'})
serie_mapping = {
    'GENERALE': 'BAC GENERAL', 'STMG': 'BAC STMG', 'STI2D': 'BAC STI2D',
    'STL': 'BAC STL', 'ST2S': 'BAC ST2S', 'STAV': 'BAC STAV',
    'STHR': 'BAC STHR', 'STD2A': 'BAC STD2A',
}
df['Série'] = df['Série'].replace(serie_mapping)

# Agrégation par groupe structurel (sans Diplôme spécialité)
GROUP_KEYS = ['Session', 'Série', 'Sexe', 'Statut du candidat', 'Libellé académie']

df = df.groupby(GROUP_KEYS, as_index=False).agg(
    présents=('Nombre de présents', 'sum'),
    admis=("Nombre d'admis totaux", 'sum')
)

df['taux_reussite'] = (df['admis'] / df['présents'] * 100).round(2)

print(f"Lignes après agrégation : {len(df):,}  (regroupement par Série × Académie × Sexe × Statut × Session)")
print(f"\nDistribution du taux de réussite :")
print(df['taux_reussite'].describe().round(2))

Lignes brutes (présents > 0) : 41,748
Lignes après agrégation : 7,477  (regroupement par Série × Académie × Sexe × Statut × Session)

Distribution du taux de réussite :
count    7477.00
mean       77.29
std        27.15
min         0.00
25%        67.14
50%        87.50
75%        97.03
max       100.00
Name: taux_reussite, dtype: float64


In [7]:
fig = make_subplots(rows=1, cols=2,
                   subplot_titles=('Distribution du taux de réussite',
                                   'Taux de réussite moyen par Série'))

fig.add_trace(
    go.Histogram(x=df['taux_reussite'], nbinsx=50,
                 marker_color='steelblue', name='Distribution'),
    row=1, col=1
)

serie_means = df.groupby('Série')['taux_reussite'].mean().reset_index().sort_values('taux_reussite')
fig.add_trace(
    go.Bar(x=serie_means['taux_reussite'], y=serie_means['Série'],
           orientation='h', marker_color='steelblue', name='Taux moyen'),
    row=1, col=2
)

fig.update_layout(height=500, showlegend=False,
                  title_text='Exploration de la variable cible')
fig.update_xaxes(title_text='Taux de réussite (%)', row=1, col=1)
fig.update_xaxes(title_text='Taux moyen (%)', row=1, col=2)
fig.update_yaxes(title_text='Nombre de groupes', row=1, col=1)
fig.show()

In [8]:
fig = make_subplots(rows=2, cols=2,
                   subplot_titles=('Par Sexe', 'Par Statut du candidat',
                                   'Par Session', 'Par Académie'))

for i, col in enumerate(['Sexe', 'Statut du candidat', 'Session', 'Libellé académie']):
    row, c = divmod(i, 2)
    means = df.groupby(col)['taux_reussite'].mean().reset_index().sort_values('taux_reussite')
    fig.add_trace(
        go.Bar(x=means['taux_reussite'], y=means[col].astype(str),
               orientation='h', marker_color='steelblue',
               name=col, showlegend=False),
        row=row+1, col=c+1
    )

fig.update_layout(height=700, title_text='Taux de réussite moyen par variable catégorielle')
fig.show()

### Corrélation variables numériques avec la cible (Session uniquement)

In [9]:
corr_session = df['Session'].corr(df['taux_reussite'])
print(f"Corrélation Session   / taux_reussite : {corr_session:.4f}")


Corrélation Session   / taux_reussite : -0.0782


## 4. Préparation du jeu de données

### Feature engineering

L'harmonisation de `Sexe` et `Série` a été faite avant l'agrégation (section 3).

**Features retenues :**
- `Série`, `Sexe`, `Statut du candidat`, `Libellé académie` — variables structurelles disponibles avant les épreuves
- `Voie` exclue : redondante avec `Série` (chaque série appartient à une voie unique)
- `Session` exclue : corrélation très faible avec le taux de réussite (-0.0782) et biais temporel en contexte de prédiction future

In [10]:
FEATURES_CAT = ['Série', 'Sexe', 'Statut du candidat', 'Libellé académie']
FEATURES_NUM = []
TARGET = 'taux_reussite'

X = df[FEATURES_CAT]
y = df[TARGET]

print(f"Shape X : {X.shape}")
print(f"Shape y : {y.shape}")
print(f"Features catégorielles : {FEATURES_CAT}")

Shape X : (7477, 4)
Shape y : (7477,)
Features catégorielles : ['Série', 'Sexe', 'Statut du candidat', 'Libellé académie']


## 5. Pipelines de preprocessing

Préprocesseur de base utilisé pour la baseline :
- Variables **catégorielles** → imputation mode + encodage One-Hot

In [11]:
preprocessor = ColumnTransformer(transformers=[
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), FEATURES_CAT)
])

print("Préprocesseur défini")

Préprocesseur défini


## 6. Modèle de référence (Baseline)

On entraîne une **régression linéaire avec toutes les features catégorielles, sans filtres**.
C'est notre jauge : chaque amélioration de preprocessing doit faire mieux que ça.

In [12]:
X_bl = df[FEATURES_CAT]
y_bl = df[TARGET]

X_tr_bl, X_te_bl, y_tr_bl, y_te_bl = train_test_split(
    X_bl, y_bl, test_size=0.2, random_state=42
)

baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

baseline_pipeline.fit(X_tr_bl, y_tr_bl)
y_pred_bl = baseline_pipeline.predict(X_te_bl)

r2_bl   = r2_score(y_te_bl, y_pred_bl)
mae_bl  = mean_absolute_error(y_te_bl, y_pred_bl)
rmse_bl = np.sqrt(mean_squared_error(y_te_bl, y_pred_bl))

print("=== Modèle de référence (Baseline) ===")
print(f"  R²   : {r2_bl:.4f}")
print(f"  MAE  : {mae_bl:.2f} pp")
print(f"  RMSE : {rmse_bl:.2f} pp")

=== Modèle de référence (Baseline) ===
  R²   : 0.4381
  MAE  : 13.26 pp
  RMSE : 20.33 pp


In [13]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Prédictions vs Valeurs réelles',
                                    'Distribution des résidus'))

fig.add_trace(
    go.Scatter(x=y_te_bl, y=y_pred_bl, mode='markers',
               marker=dict(color='steelblue', opacity=0.3, size=4),
               name='Observations'),
    row=1, col=1
)
lim_min, lim_max = float(y_te_bl.min()), float(y_te_bl.max())
fig.add_trace(
    go.Scatter(x=[lim_min, lim_max], y=[lim_min, lim_max],
               mode='lines', line=dict(color='red', dash='dash'),
               name='Prédiction parfaite'),
    row=1, col=1
)

residus_bl = y_te_bl - y_pred_bl
fig.add_trace(
    go.Histogram(x=residus_bl, nbinsx=60, marker_color='steelblue',
                 name='Résidus', showlegend=False),
    row=1, col=2
)
fig.add_vline(x=0, line_dash='dash', line_color='red', row=1, col=2)

fig.update_xaxes(title_text='Taux réel (%)', row=1, col=1)
fig.update_yaxes(title_text='Taux prédit (%)', row=1, col=1)
fig.update_xaxes(title_text='Résidu (pp)', row=1, col=2)
fig.update_yaxes(title_text='Fréquence', row=1, col=2)
fig.update_layout(
    height=450,
    title_text=f'Baseline — LinearRegression | R²={r2_bl:.4f}  MAE={mae_bl:.2f} pp'
)
fig.show()

## 7. Amélioration progressive du preprocessing

La baseline obtient un R²= 0.4557 et une MAE= 13.05 pp — c'est faible. Avant de tester des modèles plus complexes, on améliore le preprocessing étape par étape et on mesure le gain à chaque fois sur la même LinearRegression pour une comparaison équitable.

**Problème identifié** : la distribution du taux de réussite montre une forte concentration de valeurs à 0 % et 100 %,
générées par des groupes avec très peu de candidats. Ces cas extrêmes bruitent le modèle sans être représentatifs.

Étapes testées :
1. Filtre `présents >= 10`
2. Filtre `présents >= 30`
3. Filtre `>= 30` + `taux_hist` (taux de réussite historique du groupe, sessions précédentes)

In [14]:
# Calcul de taux_hist sur df complet, avant tout filtre
df = df.sort_values('Session')

df['taux_hist'] = (
    df.groupby(FEATURES_CAT)['taux_reussite']
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Fallback session 2021 : moyenne globale du groupe
global_means = df.groupby(FEATURES_CAT)['taux_reussite'].transform('mean')
df['taux_hist'] = df['taux_hist'].fillna(global_means)

print(f"Valeurs manquantes taux_hist : {df['taux_hist'].isna().sum()}")
print(f"Corrélation taux_hist / taux_reussite : {df['taux_hist'].corr(df['taux_reussite']):.4f}")

Valeurs manquantes taux_hist : 0
Corrélation taux_hist / taux_reussite : 0.7387


**Cette corrélation de 0.74 signifie que le taux historique du groupe est un très bon prédicteur du taux actuel.**

In [15]:
def evaluer_baseline(X_tr, X_te, y_tr, y_te, features_num, features_cat, label):
    """Entraîne une LinearRegression et retourne les métriques."""
    transformers = []
    if features_num:
        transformers.append(('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), features_num))
    transformers.append(('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), features_cat))

    prep = ColumnTransformer(transformers=transformers)
    pipe = Pipeline([('preprocessor', prep), ('model', LinearRegression())])
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)
    r2   = r2_score(y_te, y_pred)
    mae  = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    print(f"  {label:<25}  R²={r2:.4f}  MAE={mae:.2f}  RMSE={rmse:.2f}")
    return {'label': label, 'R²': r2, 'MAE': mae, 'RMSE': rmse}


In [16]:
etapes = []
print("Comparaison preprocessing (LinearRegression)\n")

etapes.append({'label': 'Baseline (pas de filtre)', 'R²': r2_bl, 'MAE': mae_bl, 'RMSE': rmse_bl})
print(f"  {'Baseline (pas de filtre)':<25}  R²={r2_bl:.4f}  MAE={mae_bl:.2f}  RMSE={rmse_bl:.2f}")

Comparaison preprocessing (LinearRegression)

  Baseline (pas de filtre)   R²=0.4381  MAE=13.26  RMSE=20.33


In [17]:
# Étape 1 — filtre >= 10
df_f1 = df[df['présents'] >= 10].copy()
print(f"Lignes retirées (présents < 10) : {len(df) - len(df_f1):,}  ({(1 - len(df_f1)/len(df))*100:.1f}%)")

X_f1 = df_f1[FEATURES_CAT]
y_f1 = df_f1[TARGET]
X_tr_f1, X_te_f1, y_tr_f1, y_te_f1 = train_test_split(X_f1, y_f1, test_size=0.2, random_state=42)

etapes.append(evaluer_baseline(X_tr_f1, X_te_f1, y_tr_f1, y_te_f1,
                                [], FEATURES_CAT, 'Filtre: présents >= 10'))

Lignes retirées (présents < 10) : 2,341  (31.3%)
  Filtre: présents >= 10     R²=0.7076  MAE=5.95  RMSE=9.23


In [18]:
# Étape 2 — filtre >= 30
df_f2 = df[df['présents'] >= 30].copy()
print(f"Lignes retirées (présents < 30) : {len(df) - len(df_f2):,}  ({(1 - len(df_f2)/len(df))*100:.1f}%)")

X_f2 = df_f2[FEATURES_CAT]
y_f2 = df_f2[TARGET]
X_tr_f2, X_te_f2, y_tr_f2, y_te_f2 = train_test_split(X_f2, y_f2, test_size=0.2, random_state=42)

etapes.append(evaluer_baseline(X_tr_f2, X_te_f2, y_tr_f2, y_te_f2,
                                [], FEATURES_CAT, 'Filtre: présents >= 30'))

Lignes retirées (présents < 30) : 3,704  (49.5%)
  Filtre: présents >= 30     R²=0.7394  MAE=3.92  RMSE=6.15


In [19]:
# Étape 3 — filtre >= 30 + taux_hist
X_f3 = df_f2[FEATURES_CAT + ['taux_hist']]
y_f3 = df_f2[TARGET]
X_tr_f3, X_te_f3, y_tr_f3, y_te_f3 = train_test_split(X_f3, y_f3, test_size=0.2, random_state=42)

etapes.append(evaluer_baseline(X_tr_f3, X_te_f3, y_tr_f3, y_te_f3,
                                ['taux_hist'], FEATURES_CAT, 'Filtre >= 30 + taux_hist'))

  Filtre >= 30 + taux_hist   R²=0.7962  MAE=3.42  RMSE=5.44


In [20]:
# Tableau récapitulatif des étapes
df_etapes = pd.DataFrame(etapes)
df_etapes.round(4)

,label,R²,MAE,RMSE
0,Baseline (pas de filtre),0.4381,13.2646,20.3262
1,Filtre: présents >= 10,0.7076,5.9537,9.2301
2,Filtre: présents >= 30,0.7394,3.9209,6.1547
3,Filtre >= 30 + taux_hist,0.7962,3.4160,5.4426


In [21]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Évolution du R²', 'Évolution de la MAE'))

labels = df_etapes['label'].tolist()

fig.add_trace(
    go.Scatter(x=labels, y=df_etapes['R²'], mode='lines+markers',
               marker=dict(size=10), line=dict(color='steelblue', width=2),
               name='R²'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=labels, y=df_etapes['MAE'], mode='lines+markers',
               marker=dict(size=10), line=dict(color='steelblue', width=2),
               name='MAE', showlegend=False),
    row=1, col=2
)

fig.update_yaxes(title_text='R²', row=1, col=1)
fig.update_yaxes(title_text='MAE (pp)', row=1, col=2)
fig.update_xaxes(tickangle=-15, row=1, col=1)
fig.update_xaxes(tickangle=-15, row=1, col=2)
fig.update_layout(height=450,
                  title_text='Impact de chaque étape de preprocessing sur la LinearRegression')
fig.show()

### Conclusion du preprocessing progressif

On retient la config de l'étape 3 — filtre >= 30 + taux_hist — pour la suite.
Ce preprocessing sera appliqué à tous les modèles de la comparaison.

In [22]:
df_final = df[df['présents'] >= 30].copy()

FEATURES_CAT_FINAL = ['Série', 'Sexe', 'Statut du candidat', 'Libellé académie']
FEATURES_NUM_FINAL = ['taux_hist']

X_train = df_final[FEATURES_CAT_FINAL + FEATURES_NUM_FINAL]
y_train_full = df_final[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X_train, y_train_full, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), FEATURES_NUM_FINAL),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), FEATURES_CAT_FINAL)
])

print(f"Dataset final : {len(df_final):,} lignes")
print(f"Features numériques   : {FEATURES_NUM_FINAL}")
print(f"Features catégorielles : {FEATURES_CAT_FINAL}")

Dataset final : 3,773 lignes
Features numériques   : ['taux_hist']
Features catégorielles : ['Série', 'Sexe', 'Statut du candidat', 'Libellé académie']


In [23]:
# Export du dataset final préparé pour Streamlit
df_final.to_csv('data/bac_prepared.csv', index=False)
print(f"✅ Dataset exporté : {len(df_final):,} lignes")
print(f"Colonnes : {df_final.columns.tolist()}")

✅ Dataset exporté : 3,773 lignes
Colonnes : ['Session', 'Série', 'Sexe', 'Statut du candidat', 'Libellé académie', 'présents', 'admis', 'taux_reussite', 'taux_hist']


## 8. Comparaison de plusieurs modèles via cross-validation

On évalue de façon **équitable** plusieurs modèles en utilisant la même méthode :
**K-Fold cross-validation (k=5)** sur le jeu d'entraînement.  
La régression linéaire étant déjà notre baseline, on teste ici des modèles plus complexes.  
Chaque modèle est intégré dans un Pipeline incluant le même préprocesseur.

In [24]:
modeles = {
    'Ridge':         Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost':       XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbosity=0),
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
resultats = {}

print("Évaluation par cross-validation (K=5)...\n")
for nom, modele in modeles.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', modele)])
    r2_cv   = cross_val_score(pipe, X_train, y_train, cv=kf, scoring='r2', n_jobs=-1)
    mae_cv  = -cross_val_score(pipe, X_train, y_train, cv=kf,
                                scoring='neg_mean_absolute_error', n_jobs=-1)
    rmse_cv = np.sqrt(-cross_val_score(pipe, X_train, y_train, cv=kf,
                                       scoring='neg_mean_squared_error', n_jobs=-1))
    resultats[nom] = {
        'R² moyen': r2_cv.mean(), 'R² std': r2_cv.std(),
        'MAE moyenne': mae_cv.mean(), 'RMSE moyenne': rmse_cv.mean(),
    }
    print(f"  {nom:<25}  R²={r2_cv.mean():.4f} (±{r2_cv.std():.4f})"
          f"  MAE={mae_cv.mean():.2f}  RMSE={rmse_cv.mean():.2f}")

print("\nCross-validation terminée")

Évaluation par cross-validation (K=5)...

  Ridge                      R²=0.8176 (±0.0207)  MAE=3.32  RMSE=5.20
  Random Forest              R²=0.8060 (±0.0209)  MAE=3.46  RMSE=5.37
  XGBoost                    R²=0.7777 (±0.0265)  MAE=3.59  RMSE=5.74

Cross-validation terminée


In [25]:
df_resultats = (pd.DataFrame(resultats).T
               .reset_index().rename(columns={'index': 'Modèle'})
               .sort_values('R² moyen', ascending=False))
df_resultats.round(4)

,Modèle,R² moyen,R² std,MAE moyenne,RMSE moyenne
0,Ridge,0.8176,0.0207,3.3195,5.2040
1,Random Forest,0.8060,0.0209,3.4565,5.3672
2,XGBoost,0.7777,0.0265,3.5933,5.7423


In [26]:
fig = make_subplots(rows=1, cols=2,
                   subplot_titles=('R² moyen (cross-validation)',
                                   'MAE moyenne (cross-validation)'))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
noms = df_resultats['Modèle'].values

fig.add_trace(
    go.Bar(x=noms, y=df_resultats['R² moyen'],
           error_y=dict(type='data', array=df_resultats['R² std']),
           marker_color=colors, name='R²', showlegend=False),
    row=1, col=1
)
fig.add_hline(y=r2_bl, line_dash='dash', line_color='red',
              annotation_text=f'Baseline R²={r2_bl:.3f}', row=1, col=1)

fig.add_trace(
    go.Bar(x=noms, y=df_resultats['MAE moyenne'],
           marker_color=colors, name='MAE', showlegend=False),
    row=1, col=2
)
fig.add_hline(y=mae_bl, line_dash='dash', line_color='red',
              annotation_text=f'Baseline MAE={mae_bl:.2f}', row=1, col=2)

fig.update_layout(height=450, title_text='Comparaison des modèles')
fig.update_yaxes(title_text='R²', row=1, col=1)
fig.update_yaxes(title_text='MAE (pp)', row=1, col=2)
fig.show()

## 9. Sélection du meilleur modèle

D'après la cross-validation, on sélectionne le modèle avec le meilleur R² moyen pour l'optimisation des hyperparamètres.

In [27]:
meilleur_nom = df_resultats.iloc[0]['Modèle']
print(f"Meilleur modèle sélectionné : {meilleur_nom}")

Meilleur modèle sélectionné : Ridge


## 10. Optimisation par GridSearchCV

Ridge est le meilleur modèle — il n'a qu'un seul hyperparamètre `alpha` à tuner.
On explore d'abord une grille large, puis on affine autour du meilleur alpha trouvé.

In [28]:
param_grid = {
    'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
}

pipeline_opt = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

gs = GridSearchCV(pipeline_opt, param_grid, cv=kf, scoring='r2', n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)

print(f"\nMeilleur alpha       : {gs.best_params_['model__alpha']}")
print(f"Meilleur R² (CV)     : {gs.best_score_:.4f}")

Fitting 5 folds for each of 6 candidates, totalling 30 fits

Meilleur alpha       : 1.0
Meilleur R² (CV)     : 0.8176


### Affinement autour du meilleur alpha

On resserre la grille autour du meilleur alpha trouvé pour une optimisation plus précise.

In [29]:
best_alpha = gs.best_params_['model__alpha']
param_grid_fin = {
    'model__alpha': np.logspace(
        np.log10(best_alpha) - 1,
        np.log10(best_alpha) + 1,
        20
    )
}

gs2 = GridSearchCV(pipeline_opt, param_grid_fin, cv=kf, scoring='r2', n_jobs=-1, verbose=1)
gs2.fit(X_train, y_train)

print(f"\nMeilleur alpha affiné : {gs2.best_params_['model__alpha']:.4f}")
print(f"Meilleur R² (CV)      : {gs2.best_score_:.4f}")
print(f"Gain vs grille initiale : +{gs2.best_score_ - gs.best_score_:.4f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits

Meilleur alpha affiné : 1.1288
Meilleur R² (CV)      : 0.8176
Gain vs grille initiale : +0.0000


In [30]:
# Évaluation finale sur le jeu de TEST — une seule fois
meilleur_pipeline = gs2.best_estimator_ if gs2.best_score_ > gs.best_score_ else gs.best_estimator_
meilleur_r2_cv = max(gs.best_score_, gs2.best_score_)
print(f"Modèle retenu — Ridge  alpha={meilleur_pipeline.named_steps['model'].alpha:.4f}")
print(f"R² CV : {meilleur_r2_cv:.4f}")

y_pred_final = meilleur_pipeline.predict(X_test)

r2_final   = r2_score(y_test, y_pred_final)
mae_final  = mean_absolute_error(y_test, y_pred_final)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_final))


print("RÉSULTATS FINAUX (jeu de test)")
print("══════════════════════════════════════════")
print(f"R²   : {r2_final:.4f}")
print(f"MAE  : {mae_final:.2f} points de pourcentage")
print(f"RMSE : {rmse_final:.2f} points de pourcentage")
print("══════════════════════════════════════════")
print(f"Baseline R² : {r2_bl:.4f}   gain : +{r2_final - r2_bl:.4f}")

Modèle retenu — Ridge  alpha=1.1288
R² CV : 0.8176
RÉSULTATS FINAUX (jeu de test)
══════════════════════════════════════════
R²   : 0.7960
MAE  : 3.42 points de pourcentage
RMSE : 5.45 points de pourcentage
══════════════════════════════════════════
Baseline R² : 0.4381   gain : +0.3578


### Exporter le modèle (joblib)

In [31]:
joblib.dump(meilleur_pipeline, 'model_bac.pkl')
print("✅ Modèle sauvegardé : model_bac.pkl")

✅ Modèle sauvegardé : model_bac.pkl


## 11. Analyse des prédictions

In [32]:
fig = make_subplots(rows=1, cols=2,
                   subplot_titles=('Prédictions vs Valeurs réelles',
                                   'Distribution des résidus'))

fig.add_trace(
    go.Scatter(x=y_test, y=y_pred_final, mode='markers',
               marker=dict(color='steelblue', opacity=0.3, size=4),
               name='Observations'),
    row=1, col=1
)
lim_min, lim_max = float(y_test.min()), float(y_test.max())
fig.add_trace(
    go.Scatter(x=[lim_min, lim_max], y=[lim_min, lim_max],
               mode='lines', line=dict(color='red', dash='dash'),
               name='Prédiction parfaite'),
    row=1, col=1
)

residus = y_test - y_pred_final
fig.add_trace(
    go.Histogram(x=residus, nbinsx=60, marker_color='steelblue',
                 name='Résidus', showlegend=False),
    row=1, col=2
)
fig.add_vline(x=0, line_dash='dash', line_color='red', row=1, col=2)

fig.update_xaxes(title_text='Taux réel (%)', row=1, col=1)
fig.update_yaxes(title_text='Taux prédit (%)', row=1, col=1)
fig.update_xaxes(title_text='Résidu (pp)', row=1, col=2)
fig.update_yaxes(title_text='Fréquence', row=1, col=2)

fig.update_layout(
    height=450,
    title_text=f'Modèle final — R²={r2_final:.4f}  MAE={mae_final:.2f} pp  RMSE={rmse_final:.2f} pp'
)
fig.show()

In [33]:
result = permutation_importance(
    meilleur_pipeline, X_test, y_test,
    n_repeats=10, random_state=42, n_jobs=-1
)

feat_names = FEATURES_CAT_FINAL + FEATURES_NUM_FINAL
df_perm = pd.DataFrame({
    'variable': feat_names,
    'importance': result.importances_mean
}).sort_values('importance')

fig = px.bar(
    df_perm, x='importance', y='variable',
    orientation='h',
    labels={'importance': 'Dégradation R²', 'variable': 'Variable'},
    title='Permutation importance — Ridge',
    color='importance',
    color_continuous_scale='Blues'
)
fig.update_layout(height=400, coloraxis_showscale=False,
                  yaxis={'categoryorder': 'total ascending'})
fig.show()

## 12. Bilan

### Résultats

| Étape | R² | MAE |
|---|---|---|
| Baseline (LinearRegression, sans filtre, sans taux_hist) | 0.44 | 13.26 pp |
| + Filtre présents >= 10 | 0.71 | 5.95 pp |
| + Filtre présents >= 30 | 0.74 | 3.92 pp |
| + taux_hist | 0.80 | 3.42 pp |
| **Modèle final — Ridge optimisé** | **0.82** | **3.32 pp** |

---

### Récap du process

**1. Agrégation** — Sans `Diplôme spécialité`, plusieurs lignes représentaient le même groupe structurel. On a agrégé par `Série × Académie × Sexe × Statut × Session` en sommant présents et admis avant de recalculer le taux (pondération correcte par taille de groupe).

**2. Features retenues** — `Série`, `Sexe`, `Statut du candidat`, `Libellé académie` :
- `Voie` exclue : entièrement redondante avec `Série` (heatmap — chaque série appartient à une seule voie)
- `Session` exclue : voir ci-dessous

**3. Pourquoi taux_hist plutôt que Session** — `Session` capte une tendance temporelle globale (ex. 2023 < 2021) mais n'apporte aucune information sur le groupe lui-même. `taux_hist` capte l'historique propre à chaque groupe (`Série × Académie × Sexe × Statut`) — corrélation 0.74 avec la cible vs -0.0782 pour Session. De plus, en contexte Streamlit, un utilisateur prédit pour une session future : Session serait une valeur inconnue, `taux_hist` reste calculable.

**4. Filtre présents >= 30** — Les groupes de moins de 30 présents génèrent des taux artificiels à 0 % ou 100 % qui bruitent le modèle sans être représentatifs. Le filtre supprime ~60% des lignes mais améliore considérablement le signal.

**5. Pourquoi Ridge** — Le signal dominant (`taux_hist`, corrélation 0.74) est très linéaire. Ridge l'exploite directement. La permutation importance confirme : `taux_hist` explique à lui seul ~65% de la dégradation du R² quand on le retire — `Statut du candidat` est second (~22%), `Série`, `Académie` et `Sexe` ont un apport marginal mais non nul.
